# 🧠 记忆增强模型训练

使用 Google Colab **免费版 T4 GPU** 训练 100M 参数的记忆增强模型

**预计时间**: 2-4 小时
**GPU**: T4 (16GB)
**费用**: 完全免费 ✅

## Step 1: 准备工作

In [ ]:
# 克隆代码仓库
!git clone https://github.com/ZhaoDongdong0109/new-agent-memory.git
%cd new-agent-memory
!git pull origin performance-optimizations

## Step 2: 安装依赖

In [ ]:
# 安装 PyTorch 和相关库
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers datasets accelerate wandb tqdm openai

## Step 3: 设置 API 密钥（可选）

如果你想用 LLM 智能分析训练状态，设置 API 密钥：

In [ ]:
# 可选：设置 OpenAI API 密钥（用于 LLM 智能分析）
# import os
# os.environ['OPENAI_API_KEY'] = 'your-api-key-here'

# 或使用 DeepSeek（更便宜）
# os.environ['DEEPSEEK_API_KEY'] = 'your-api-key-here'

print("API 密钥设置完成（跳过）")

## Step 4: 生成训练数据

In [ ]:
import sys
sys.path.insert(0, '/content/new-agent-memory')

from data_generator import TrainingDataGenerator

print("开始生成训练数据...")
generator = TrainingDataGenerator()

# 生成对话训练数据
training_data = generator.generate_dataset(
    num_samples=1000, 
    output_file="training_data.json"
)

# 生成检索数据
retrieval_data = generator.generate_retrieval_pairs(
    num_pairs=500, 
    output_file="retrieval_data.json"
)

print(f"✅ 生成完成！")
print(f"   对话样本: {len(training_data)}")
print(f"   检索对: {len(retrieval_data)}")

## Step 5: 测试模型

In [ ]:
import torch

# 检查 GPU
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# 测试模型
from memory_model import create_model, ModelConfig

# 针对 T4 优化配置
config = ModelConfig(
    encoder_hidden_size=256,
    encoder_num_layers=6,
    encoder_num_heads=4,
    adapter_hidden_size=512,
    adapter_num_layers=12,
    adapter_num_heads=8,
    batch_size=8  # T4 显存限制
)

model = create_model(config)
model = model.to('cuda')

print("\n✅ 模型创建成功！")

## Step 6: 开始训练

⚠️ **这将需要 2-4 小时**

训练会自动进行，包括：
- 编码器训练
- 检索器训练
- 适配器训练

In [ ]:
from train import train_full_model, TrainingConfig

# 训练配置
config = TrainingConfig(
    batch_size=8,  # T4 优化
    learning_rate=1e-4,
    num_epochs=10,
    device='cuda'
)

# 开始训练
print("=" * 70)
print("开始训练记忆增强模型")
print("预计时间: 2-4 小时")
print("=" * 70)

model = train_full_model(config)

print("\n" + "=" * 70)
print("✅ 训练完成！")
print("=" * 70)

## Step 7: 测试模型效果

In [ ]:
# 测试记忆编码
print("测试记忆编码...")

test_text = "我叫张三，我喜欢编程"
input_ids = torch.randint(0, 50000, (1, 32)).to('cuda')

with torch.no_grad():
    memory_vector, importance = model.encoder(input_ids)
    print(f"记忆向量形状: {memory_vector.shape}")
    print(f"重要性分数: {importance.item():.4f}")

print("\n✅ 模型测试成功！")

## 🎉 完成！

训练已完成！你现在可以：

1. **下载模型** - 从左侧文件栏下载 `checkpoints/` 目录
2. **查看报告** - 训练报告在 `training_reports/` 目录
3. **继续实验** - 修改代码后重新训练